# Variant Impact ML Features - Exploratory Data Analysis
**DNA Gene Mapping Project - ML Phase V5**  
**Author:** Sharique Mohammad  
**Date:** February 2026  
**Table:** variant_impact_ml_features (4.1M variants, 103 columns)

## Objective
Deep exploration of variant functional impact: conservation scores, protein domain effects, LOF categories, splice impacts, and gene-level impact burden.

## Key Questions
1. What is the distribution of variant impact tiers?
2. How do conservation scores (PhyloP, PhastCons, GERP, CADD) relate to pathogenicity?
3. What protein domains are most commonly affected?
4. What is the LOF landscape across variants?
5. Which gene-level impact features best predict high-impact variants?

## Deliverables
- 12+ visualizations saved to variant_impact_ml_features/images/
- EDA report saved to variant_impact_ml_features/reports/
- Missing values, correlation matrix, feature statistics saved to variant_impact_ml_features/metrics/

## 1. Setup and Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine
import os
from pathlib import Path
from dotenv import load_dotenv
import warnings
warnings.filterwarnings('ignore')

load_dotenv()

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

PROJECT_ROOT = Path().absolute().parent.parent
BASE_OUT     = PROJECT_ROOT / 'data' / 'analytical' / 'variant_impact_ml_features'
IMAGES_DIR   = BASE_OUT / 'images'
REPORTS_DIR  = BASE_OUT / 'reports'
METRICS_DIR  = BASE_OUT / 'metrics'

IMAGES_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)

print("Setup complete")
print(f"Images  : {IMAGES_DIR}")
print(f"Reports : {REPORTS_DIR}")
print(f"Metrics : {METRICS_DIR}")

## 2. Database Connection

In [ ]:
POSTGRES_HOST     = os.getenv("POSTGRES_HOST")
POSTGRES_PORT     = os.getenv("POSTGRES_PORT")
POSTGRES_DB       = os.getenv("POSTGRES_DB")
POSTGRES_USER     = os.getenv("POSTGRES_USER")
POSTGRES_PASSWORD = os.getenv("POSTGRES_PASSWORD")

conn_str = f"postgresql://{POSTGRES_USER}:{POSTGRES_PASSWORD}@{POSTGRES_HOST}:{POSTGRES_PORT}/{POSTGRES_DB}"
engine = create_engine(conn_str)

print("Database connection established")
print(f"Host : {POSTGRES_HOST}:{POSTGRES_PORT}")
print(f"DB   : {POSTGRES_DB}")

## 3. Data Loading

In [ ]:
print("Loading variant_impact_ml_features (10% sample)...")
query = """
    SELECT * FROM gold.variant_impact_ml_features
    TABLESAMPLE SYSTEM (10)
"""
df = pd.read_sql(query, engine)

print(f"Rows loaded    : {len(df):,}")
print(f"Columns        : {len(df.columns)}")
print(f"Memory usage   : {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
print(f"Full table est : ~4.1M rows")

## 4. Type Conversion

In [ ]:
# INT columns from gold schema
int_cols = [
    'review_quality_score', 'domain_count', 'domain_type_count',
    'mutation_severity_score', 'pathogenicity_score', 'combined_impact_score',
    'conservation_level', 'tissues_expressed_count', 'cancer_mutation_count',
    'disease_count', 'gene_total_variants', 'gene_high_impact_count',
    'gene_very_high_impact_count', 'gene_lof_count', 'gene_splice_variant_count',
    'gene_domain_affecting_count', 'gene_max_impact_score'
]

# DOUBLE columns from gold schema
double_cols = [
    'phylop_score', 'phastcons_score', 'gerp_score', 'cadd_phred',
    'druggability_score', 'max_expression_tpm', 'gene_avg_impact_score'
]

# BOOLEAN columns from gold schema
bool_cols = [
    'is_pathogenic', 'is_benign', 'is_vus',
    'is_missense_variant', 'is_frameshift_variant', 'is_nonsense_variant',
    'is_splice_variant', 'is_snv', 'is_insertion', 'is_deletion',
    'has_functional_domain', 'has_zinc_finger', 'has_kinase_domain',
    'has_receptor_domain', 'has_sh2_domain', 'has_sh3_domain', 'has_ph_domain',
    'affects_functional_domain', 'has_multiple_domain_types',
    'is_highly_conserved', 'is_constrained', 'is_likely_deleterious',
    'is_high_impact', 'is_very_high_impact', 'is_conservation_constrained',
    'is_highly_conserved_region', 'is_domain_affecting', 'is_loss_of_function',
    'is_splice_affecting', 'has_cadd_score', 'is_deleterious_by_cadd',
    'is_splice_site_variant', 'is_kinase', 'is_receptor', 'is_enzyme',
    'is_pharmacogene', 'is_druggable_gene', 'is_key_protein_type',
    'is_well_annotated', 'is_broadly_expressed', 'is_highly_expressed',
    'is_cancer_gene', 'is_cancer_relevant_variant',
    'has_cancer_disease', 'has_neurological_disease',
    'has_metabolic_disease', 'has_cardiovascular_disease',
    'is_disease_associated_gene'
]

for col in int_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').astype('Int64')

for col in double_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

for col in bool_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).str.lower().map({'true': True, 'false': False})

print("Type conversion complete")
print(df.dtypes.value_counts())

## 5. Dataset Overview

In [ ]:
total = len(df)

print("Dataset Overview")
print("=" * 60)
print(f"Total variants  : {total:,}")
print(f"Total columns   : {len(df.columns)}")
print(f"Unique genes    : {df['gene_name'].nunique():,}")
print(f"Chromosomes     : {df['chromosome'].nunique()}")
print()
print("Column list:")
for i, col in enumerate(df.columns, 1):
    print(f"  {i:2d}. {col:<55} {str(df[col].dtype)}")

## 6. Missing Values Analysis

In [ ]:
missing = pd.DataFrame({
    'column': df.columns,
    'missing_count': df.isnull().sum().values,
    'missing_pct': (df.isnull().sum().values / len(df) * 100).round(2)
}).sort_values('missing_pct', ascending=False)

missing_with_nulls = missing[missing['missing_count'] > 0]
print(f"Columns with missing values: {len(missing_with_nulls)}")
print()
print(missing_with_nulls.to_string(index=False))

missing.to_csv(METRICS_DIR / 'missing_values.csv', index=False)
print(f"\nSaved: {METRICS_DIR / 'missing_values.csv'}")

## 7. Target Variable Analysis

In [ ]:
pathogenic_count = int(df['is_pathogenic'].sum()) if 'is_pathogenic' in df.columns else 0
benign_count     = int(df['is_benign'].sum())     if 'is_benign'     in df.columns else 0
vus_count        = int(df['is_vus'].sum())        if 'is_vus'        in df.columns else 0
high_impact      = int(df['is_high_impact'].sum()) if 'is_high_impact' in df.columns else 0
very_high_impact = int(df['is_very_high_impact'].sum()) if 'is_very_high_impact' in df.columns else 0

print("Target Variable Distribution")
print("=" * 50)
print(f"Pathogenic      : {pathogenic_count:>10,}  ({pathogenic_count/total*100:>5.1f}%)")
print(f"Benign          : {benign_count:>10,}  ({benign_count/total*100:>5.1f}%)")
print(f"VUS             : {vus_count:>10,}  ({vus_count/total*100:>5.1f}%)")
print(f"High Impact     : {high_impact:>10,}  ({high_impact/total*100:>5.1f}%)")
print(f"Very High Impact: {very_high_impact:>10,}  ({very_high_impact/total*100:>5.1f}%)")

if pathogenic_count > 0 and benign_count > 0:
    imbalance_ratio = max(pathogenic_count, benign_count) / min(pathogenic_count, benign_count)
    print(f"\nClass imbalance ratio : {imbalance_ratio:.2f}:1")
    print(f"SMOTE needed          : {imbalance_ratio > 5}")

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

categories = ['Pathogenic', 'Benign', 'VUS']
counts     = [pathogenic_count, benign_count, vus_count]
colors     = ['#e74c3c', '#27ae60', '#95a5a6']

bars = axes[0].bar(categories, counts, color=colors, alpha=0.8, edgecolor='black')
for bar, count in zip(bars, counts):
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height(),
                 f'{count:,}\n({count/total*100:.1f}%)',
                 ha='center', va='bottom', fontsize=10, fontweight='bold')
axes[0].set_ylabel('Number of Variants', fontsize=11, fontweight='bold')
axes[0].set_title('Pathogenicity Distribution', fontsize=12, fontweight='bold')
axes[0].grid(axis='y', alpha=0.3)

impact_cats = ['High Impact', 'Very High Impact']
impact_vals = [high_impact, very_high_impact]
axes[1].bar(impact_cats, impact_vals, color=['#f39c12', '#e74c3c'], alpha=0.8, edgecolor='black')
for i, (cat, val) in enumerate(zip(impact_cats, impact_vals)):
    axes[1].text(i, val, f'{val:,}\n({val/total*100:.1f}%)',
                 ha='center', va='bottom', fontsize=10, fontweight='bold')
axes[1].set_ylabel('Number of Variants', fontsize=11, fontweight='bold')
axes[1].set_title('Impact Level Distribution', fontsize=12, fontweight='bold')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(IMAGES_DIR / '01_target_variable.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: 01_target_variable.png")

## 8. Variant Impact Tier

In [ ]:
if 'variant_impact_tier' in df.columns:
    print("Variant Impact Tier Distribution")
    tier_dist = df['variant_impact_tier'].value_counts()
    print(tier_dist.to_string())

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    tier_dist.sort_values().plot(kind='barh', ax=axes[0], color='steelblue', alpha=0.8, edgecolor='black')
    axes[0].set_xlabel('Number of Variants', fontsize=11, fontweight='bold')
    axes[0].set_title('Variant Impact Tier Distribution', fontsize=12, fontweight='bold')
    axes[0].grid(axis='x', alpha=0.3)

    if 'lof_category' in df.columns:
        lof_dist = df['lof_category'].value_counts()
        lof_dist.sort_values().plot(kind='barh', ax=axes[1], color='coral', alpha=0.8, edgecolor='black')
        axes[1].set_xlabel('Number of Variants', fontsize=11, fontweight='bold')
        axes[1].set_title('LOF Category Distribution', fontsize=12, fontweight='bold')
        axes[1].grid(axis='x', alpha=0.3)

    plt.tight_layout()
    plt.savefig(IMAGES_DIR / '02_impact_tier_lof.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: 02_impact_tier_lof.png")

## 9. Conservation Scores

In [ ]:
conservation_cols = ['phylop_score', 'phastcons_score', 'gerp_score', 'cadd_phred']
conservation_cols = [c for c in conservation_cols if c in df.columns]

print("Conservation Score Summary")
print("=" * 50)
for col in conservation_cols:
    data = df[col].dropna()
    print(f"  {col:<20} : min={data.min():.3f}  median={data.median():.3f}  max={data.max():.3f}  missing={df[col].isnull().sum():,}")

n_cols = 2
n_rows = (len(conservation_cols) + 1) // 2
fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, n_rows * 5))
axes = axes.flatten()

thresholds = {'phylop_score': 2.7, 'phastcons_score': 0.7, 'gerp_score': 2.0, 'cadd_phred': 20}

for i, col in enumerate(conservation_cols):
    data = df[col].dropna()
    axes[i].hist(data, bins=50, color='darkgreen', alpha=0.8, edgecolor='black')
    if col in thresholds:
        axes[i].axvline(thresholds[col], color='red', linestyle='--', linewidth=2,
                        label=f'Threshold: {thresholds[col]}')
    axes[i].set_title(col, fontsize=11, fontweight='bold')
    axes[i].set_xlabel('Score', fontsize=9)
    axes[i].set_ylabel('Frequency', fontsize=9)
    if col in thresholds:
        axes[i].legend(fontsize=9)
    axes[i].grid(alpha=0.3)

for i in range(len(conservation_cols), len(axes)):
    axes[i].axis('off')

plt.tight_layout()
plt.savefig(IMAGES_DIR / '03_conservation_scores.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: 03_conservation_scores.png")

In [ ]:
if 'phylop_score' in df.columns and 'is_high_impact' in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    for val, label, color in [(True, 'High Impact', '#e74c3c'), (False, 'Not High Impact', '#27ae60')]:
        subset = df[df['is_high_impact'] == val]['phylop_score'].dropna()
        axes[0].hist(subset, bins=50, alpha=0.6, label=label, color=color, edgecolor='black')
    axes[0].set_xlabel('PhyloP Score', fontsize=11, fontweight='bold')
    axes[0].set_title('PhyloP Score by Impact Level', fontsize=12, fontweight='bold')
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    if 'cadd_phred' in df.columns:
        for val, label, color in [(True, 'High Impact', '#e74c3c'), (False, 'Not High Impact', '#27ae60')]:
            subset = df[df['is_high_impact'] == val]['cadd_phred'].dropna()
            axes[1].hist(subset, bins=50, alpha=0.6, label=label, color=color, edgecolor='black')
        axes[1].set_xlabel('CADD Phred Score', fontsize=11, fontweight='bold')
        axes[1].set_title('CADD Score by Impact Level', fontsize=12, fontweight='bold')
        axes[1].legend()
        axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(IMAGES_DIR / '04_conservation_by_impact.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: 04_conservation_by_impact.png")

## 10. Protein Domain Analysis

In [ ]:
domain_flags = [
    'has_functional_domain', 'has_zinc_finger', 'has_kinase_domain',
    'has_receptor_domain', 'has_sh2_domain', 'has_sh3_domain', 'has_ph_domain',
    'affects_functional_domain', 'has_multiple_domain_types', 'is_domain_affecting'
]
domain_flags = [c for c in domain_flags if c in df.columns]

domain_counts = {col: int(df[col].sum()) for col in domain_flags}

print("Protein Domain Features")
print("=" * 50)
for col, count in sorted(domain_counts.items(), key=lambda x: -x[1]):
    print(f"  {col:<35} : {count:>10,}  ({count/total*100:.1f}%)")

fig, ax = plt.subplots(figsize=(12, 8))
names  = [c.replace('has_', '').replace('is_', '').replace('affects_', '').replace('_', ' ').title() for c in domain_flags]
values = [domain_counts[c] for c in domain_flags]
colors = plt.cm.Set2(np.linspace(0, 1, len(names)))

bars = ax.barh(names, values, color=colors, alpha=0.8, edgecolor='black')
for bar, val in zip(bars, values):
    ax.text(bar.get_width() + max(values)*0.01,
            bar.get_y() + bar.get_height()/2.,
            f'{val/total*100:.1f}%', va='center', fontsize=9)
ax.set_xlabel('Count', fontsize=11, fontweight='bold')
ax.set_title('Protein Domain Feature Distribution', fontsize=12, fontweight='bold')
ax.set_xlim(0, max(values) * 1.2)
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig(IMAGES_DIR / '05_protein_domain_features.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: 05_protein_domain_features.png")

if 'domain_count' in df.columns:
    print("\nDomain Count Statistics:")
    print(df['domain_count'].describe().to_string())

## 11. Variant Type Distribution

In [ ]:
variant_flags = [
    'is_missense_variant', 'is_frameshift_variant', 'is_nonsense_variant',
    'is_splice_variant', 'is_snv', 'is_insertion', 'is_deletion',
    'is_splice_site_variant', 'is_splice_affecting', 'is_loss_of_function'
]
variant_flags = [c for c in variant_flags if c in df.columns]

var_counts = {col: int(df[col].sum()) for col in variant_flags}

fig, axes = plt.subplots(1, 2, figsize=(14, 7))

names  = [c.replace('is_', '').replace('_variant', '').replace('_', ' ').title() for c in variant_flags]
values = [var_counts[c] for c in variant_flags]
colors = plt.cm.Set3(np.linspace(0, 1, len(names)))

bars = axes[0].barh(names, values, color=colors, alpha=0.8, edgecolor='black')
for bar, val in zip(bars, values):
    axes[0].text(bar.get_width() + max(values)*0.01,
                 bar.get_y() + bar.get_height()/2.,
                 f'{val/total*100:.1f}%', va='center', fontsize=9)
axes[0].set_xlabel('Count', fontsize=11, fontweight='bold')
axes[0].set_title('Variant Type Distribution', fontsize=12, fontweight='bold')
axes[0].set_xlim(0, max(values) * 1.2)
axes[0].grid(axis='x', alpha=0.3)

if 'variant_type' in df.columns:
    vtype_dist = df['variant_type'].value_counts().head(10)
    vtype_dist.sort_values().plot(kind='barh', ax=axes[1], color='steelblue', alpha=0.8, edgecolor='black')
    axes[1].set_xlabel('Count', fontsize=11, fontweight='bold')
    axes[1].set_title('Variant Type Categories', fontsize=12, fontweight='bold')
    axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig(IMAGES_DIR / '06_variant_types.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: 06_variant_types.png")

## 12. Pathogenicity and Impact Scores

In [ ]:
score_cols = [
    'pathogenicity_score', 'mutation_severity_score', 'combined_impact_score',
    'conservation_level', 'gene_avg_impact_score', 'gene_max_impact_score'
]
score_cols = [c for c in score_cols if c in df.columns]

n_cols = 3
n_rows = (len(score_cols) + 2) // 3
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, n_rows * 4))
axes = axes.flatten()

for i, col in enumerate(score_cols):
    data = df[col].dropna()
    axes[i].hist(data, bins=40, color='steelblue', alpha=0.8, edgecolor='black')
    median_val = data.median()
    axes[i].axvline(median_val, color='red', linestyle='--', linewidth=1.5,
                    label=f'Median: {median_val:.1f}')
    axes[i].set_title(col, fontsize=10, fontweight='bold')
    axes[i].set_xlabel('Value', fontsize=9)
    axes[i].set_ylabel('Frequency', fontsize=9)
    axes[i].legend(fontsize=8)
    axes[i].grid(alpha=0.3)

for i in range(len(score_cols), len(axes)):
    axes[i].axis('off')

plt.tight_layout()
plt.savefig(IMAGES_DIR / '07_impact_scores.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: 07_impact_scores.png")

## 13. Gene-Level Impact Features

In [ ]:
gene_impact_cols = [
    'gene_total_variants', 'gene_high_impact_count', 'gene_very_high_impact_count',
    'gene_lof_count', 'gene_splice_variant_count', 'gene_domain_affecting_count'
]
gene_impact_cols = [c for c in gene_impact_cols if c in df.columns]

if gene_impact_cols:
    n_cols = 3
    n_rows = (len(gene_impact_cols) + 2) // 3
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, n_rows * 4))
    axes = axes.flatten()

    for i, col in enumerate(gene_impact_cols):
        data = df[col].dropna()
        axes[i].hist(data, bins=40, color='mediumpurple', alpha=0.8, edgecolor='black')
        axes[i].set_title(col, fontsize=10, fontweight='bold')
        axes[i].set_xlabel('Count', fontsize=9)
        axes[i].set_ylabel('Frequency', fontsize=9)
        axes[i].grid(alpha=0.3)

    for i in range(len(gene_impact_cols), len(axes)):
        axes[i].axis('off')

    plt.tight_layout()
    plt.savefig(IMAGES_DIR / '08_gene_impact_features.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: 08_gene_impact_features.png")

if 'gene_lof_tolerance' in df.columns:
    print("\nGene LOF Tolerance Distribution")
    lof_tol = df['gene_lof_tolerance'].value_counts()
    print(lof_tol.to_string())

## 14. Disease and Cancer Context

In [ ]:
disease_cancer_flags = [
    'is_cancer_gene', 'is_cancer_relevant_variant',
    'has_cancer_disease', 'has_neurological_disease',
    'has_metabolic_disease', 'has_cardiovascular_disease',
    'is_disease_associated_gene'
]
disease_cancer_flags = [c for c in disease_cancer_flags if c in df.columns]

dc_counts = {col: int(df[col].sum()) for col in disease_cancer_flags}

print("Disease and Cancer Context")
print("=" * 50)
for col, count in sorted(dc_counts.items(), key=lambda x: -x[1]):
    print(f"  {col:<40} : {count:>10,}  ({count/total*100:.1f}%)")

fig, ax = plt.subplots(figsize=(12, 7))
names  = [c.replace('_', ' ').title() for c in disease_cancer_flags]
values = [dc_counts[c] for c in disease_cancer_flags]
colors = plt.cm.Set1(np.linspace(0, 1, len(names)))

bars = ax.barh(names, values, color=colors, alpha=0.8, edgecolor='black')
for bar, val in zip(bars, values):
    ax.text(bar.get_width() + max(values)*0.01,
            bar.get_y() + bar.get_height()/2.,
            f'{val/total*100:.1f}%', va='center', fontsize=9)
ax.set_xlabel('Count', fontsize=11, fontweight='bold')
ax.set_title('Disease and Cancer Context Features', fontsize=12, fontweight='bold')
ax.set_xlim(0, max(values) * 1.2)
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig(IMAGES_DIR / '09_disease_cancer_context.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: 09_disease_cancer_context.png")

## 15. Correlation Analysis

In [ ]:
corr_features = [
    'review_quality_score', 'domain_count', 'domain_type_count',
    'mutation_severity_score', 'pathogenicity_score', 'combined_impact_score',
    'conservation_level', 'tissues_expressed_count', 'cancer_mutation_count',
    'disease_count', 'gene_total_variants', 'gene_high_impact_count',
    'gene_lof_count', 'gene_max_impact_score',
    'phylop_score', 'phastcons_score', 'gerp_score', 'cadd_phred',
    'druggability_score', 'gene_avg_impact_score'
]
corr_features = [c for c in corr_features if c in df.columns]

corr_data   = df[corr_features].apply(pd.to_numeric, errors='coerce')
corr_matrix = corr_data.corr()

corr_matrix.to_csv(METRICS_DIR / 'correlation_matrix.csv')
print(f"Saved: {METRICS_DIR / 'correlation_matrix.csv'}")

fig, ax = plt.subplots(figsize=(18, 16))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=0.5,
            cbar_kws={'shrink': 0.8}, ax=ax, annot_kws={'size': 7})
ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig(IMAGES_DIR / '10_correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: 10_correlation_matrix.png")

print("\nHighly correlated pairs (|r| > 0.9):")
found = False
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        val = corr_matrix.iloc[i, j]
        if abs(val) > 0.9:
            print(f"  {corr_matrix.columns[i]} vs {corr_matrix.columns[j]}: {val:.3f}")
            found = True
if not found:
    print("  None found above 0.9")

## 16. Feature Statistics Summary

In [ ]:
stats = df[corr_features].describe().T
stats['missing_pct'] = (df[corr_features].isnull().sum() / len(df) * 100).values
stats.to_csv(METRICS_DIR / 'feature_statistics.csv')
print(f"Saved: {METRICS_DIR / 'feature_statistics.csv'}")
print()
print(stats.to_string())

## 17. EDA Report

In [ ]:
report_path      = REPORTS_DIR / 'variant_impact_eda_report.txt'
images_generated = sorted(IMAGES_DIR.glob('*.png'))

with open(report_path, 'w') as f:
    f.write("=" * 80 + "\n")
    f.write("VARIANT IMPACT ML FEATURES - EDA REPORT\n")
    f.write("DNA Gene Mapping Project - ML Phase V5\n")
    f.write("=" * 80 + "\n\n")

    f.write("TABLE: gold.variant_impact_ml_features\n")
    f.write(f"Sample rows  : {len(df):,} (10% of ~4.1M)\n")
    f.write(f"Columns      : {len(df.columns)}\n")
    f.write(f"Unique genes : {df['gene_name'].nunique():,}\n\n")

    f.write("TARGET VARIABLE (is_high_impact)\n")
    f.write("-" * 40 + "\n")
    f.write(f"Pathogenic      : {pathogenic_count:,} ({pathogenic_count/total*100:.1f}%)\n")
    f.write(f"Benign          : {benign_count:,} ({benign_count/total*100:.1f}%)\n")
    f.write(f"VUS             : {vus_count:,} ({vus_count/total*100:.1f}%)\n")
    f.write(f"High Impact     : {high_impact:,} ({high_impact/total*100:.1f}%)\n")
    f.write(f"Very High Impact: {very_high_impact:,} ({very_high_impact/total*100:.1f}%)\n")
    if pathogenic_count > 0 and benign_count > 0:
        f.write(f"Imbalance  : {imbalance_ratio:.2f}:1\n")
        f.write(f"SMOTE      : {'Recommended' if imbalance_ratio > 5 else 'Not required'}\n\n")

    f.write("CONSERVATION SCORES AVAILABLE\n")
    f.write("-" * 40 + "\n")
    for col in conservation_cols:
        pct_missing = df[col].isnull().sum() / total * 100
        f.write(f"  {col:<20} : {pct_missing:.1f}% missing\n")
    f.write("\n")

    f.write("MISSING VALUES\n")
    f.write("-" * 40 + "\n")
    f.write(f"Columns with missing data: {len(missing_with_nulls)}\n")
    if len(missing_with_nulls) > 0:
        for _, row in missing_with_nulls.head(10).iterrows():
            f.write(f"  {row['column']:<55} {row['missing_pct']:.1f}%\n")
    f.write("\n")

    f.write("VISUALIZATIONS GENERATED\n")
    f.write("-" * 40 + "\n")
    for img in images_generated:
        f.write(f"  {img.name}\n")
    f.write("\n")

    f.write("OUTPUT FILES\n")
    f.write("-" * 40 + "\n")
    f.write(f"  images/  : {len(images_generated)} PNG files\n")
    f.write(f"  reports/ : variant_impact_eda_report.txt\n")
    f.write(f"  metrics/ : missing_values.csv, correlation_matrix.csv, feature_statistics.csv\n")
    f.write("\n")

    f.write("NEXT STEPS\n")
    f.write("-" * 40 + "\n")
    f.write("  1. Note conservation score missingness before training\n")
    f.write("  2. Remove leakage columns: clinical_significance_simple, clinvar_pathogenicity_class, review_status\n")
    f.write("  3. Apply correlation filter (r > 0.95)\n")
    f.write("  4. Proceed to 05_structural_variant_ml_features_eda.ipynb\n")

print(f"Saved: {report_path}")
print()
print("=" * 60)
print("VARIANT IMPACT ML FEATURES EDA COMPLETE")
print("=" * 60)
print(f"  Images   : {len(images_generated)}")
print(f"  Reports  : 1")
print(f"  Metrics  : 3 CSV files")
print(f"  Output   : {BASE_OUT}")
print()
print("Next: 05_structural_variant_ml_features_eda.ipynb")